# Notebook 12 - Array Functions

**Dataset:** `samples.bakehouse.sales_transactions` + `sales_franchises` + `samples.wanderbricks.reviews`  
**Difficulty:** Medium  
**Topics:** `collect_list`, `collect_set`, `array_contains`, `explode`, `posexplode`, `array_distinct`, `flatten`, `array_sort`, `size`, `element_at`, `array_union`, `array_intersect`, `array_except`, `arrays_zip`, `transform`


In [ ]:
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()

transactions = spark.table("samples.bakehouse.sales_transactions")
franchises   = spark.table("samples.bakehouse.sales_franchises")
reviews      = spark.table("samples.wanderbricks.reviews")


## Problem 1 - Collect Products per Franchise

Join `sales_transactions` with `sales_franchises` on `franchiseID`.  
Use `F.collect_set("product")` grouped by `franchiseID` and `name` to get unique products per franchise.  
Add a `product_count` using `F.size()`.  
**Columns:** `franchiseID`, `name`, `products_sold`, `product_count`


In [ ]:
result_1 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 1 ─────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'franchiseid' in cols, "Missing: franchiseID"
assert 'products_sold' in cols, "Missing: products_sold"
assert 'product_count' in cols, "Missing: product_count"
cnt = result_1.count()
assert cnt > 0, "Got 0 rows"
# product_count should equal size of products_sold array
mismatch = result_1.filter(F.col("product_count") != F.size(F.col("products_sold")))
assert mismatch.count() == 0, "product_count does not match size of products_sold"
print(f"Problem 1 passed ✓  ({cnt} rows)")


## Problem 2 - Explode Collected Array

Starting from the franchise-products array (rebuild from `result_1` or rebuild independently),  
use `F.explode("products_sold")` to expand the array back to one row per product per franchise.  
**Columns:** `franchiseID`, `name`, `product`


In [ ]:
result_2 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 2 ─────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'franchiseid' in cols, "Missing: franchiseID"
assert 'name' in cols, "Missing: name"
assert 'product' in cols, "Missing: product"
cnt = result_2.count()
assert cnt > 0, "Got 0 rows"
# exploded result should have more rows than the array result (one per unique product)
assert cnt >= result_1.count(), "Exploded result should have >= rows than franchise count"
print(f"Problem 2 passed ✓  ({cnt} rows)")


## Problem 3 - Array Contains Check

On the franchise-products array, use `F.array_contains(F.col("products_sold"), "Coffee")` to flag franchises that sell Coffee.  
**Columns:** `franchiseID`, `name`, `sells_coffee`, `product_count`


In [ ]:
result_3 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 3 ─────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'franchiseid' in cols, "Missing: franchiseID"
assert 'sells_coffee' in cols, "Missing: sells_coffee"
assert 'product_count' in cols, "Missing: product_count"
cnt = result_3.count()
assert cnt > 0, "Got 0 rows"
# sells_coffee should be boolean
assert result_3.schema["sells_coffee"].dataType.typeName() in ("boolean",), "sells_coffee must be BooleanType"
print(f"Problem 3 passed ✓  ({cnt} rows)")


## Problem 4 - Array Set Operations

Split `sales_transactions` into first half and second half by `dateTime` (use the median date as the split point).  
Build product arrays per franchise for each half. Then use `F.array_intersect` and `F.array_except` to find common products and products dropped in the second half.  
**Columns:** `franchiseID`, `first_half_products`, `second_half_products`, `common_products`, `dropped_products`


In [ ]:
result_4 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 4 ─────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'franchiseid' in cols, "Missing: franchiseID"
assert 'first_half_products' in cols, "Missing: first_half_products"
assert 'second_half_products' in cols, "Missing: second_half_products"
assert 'common_products' in cols, "Missing: common_products"
assert 'dropped_products' in cols, "Missing: dropped_products"
cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
# common_products should be a subset of first_half_products (size check)
size_check = result_4.filter(F.size(F.col("common_products")) > F.size(F.col("first_half_products")))
assert size_check.count() == 0, "common_products cannot be larger than first_half_products"
print(f"Problem 4 passed ✓  ({cnt} rows)")


## Problem 5 - posexplode for Indexed Arrays

Use `F.collect_list("product")` per `customerID` ordered by `dateTime` to build a purchase sequence.  
Then use `F.posexplode("purchase_sequence")` to show each customer's purchase history with its index position.  
**Columns:** `customerID`, `purchase_position`, `product`


In [ ]:
result_5 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 5 ─────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'customerid' in cols, "Missing: customerID"
assert 'purchase_position' in cols, "Missing: purchase_position"
assert 'product' in cols, "Missing: product"
cnt = result_5.count()
assert cnt > 0, "Got 0 rows"
# purchase_position should start from 0
min_pos = result_5.agg(F.min("purchase_position")).collect()[0][0]
assert min_pos == 0, "purchase_position should start from 0"
print(f"Problem 5 passed ✓  ({cnt} rows)")


## Problem 6 - Array Sort and Element Access

Collect a sorted product list per franchise using `F.array_sort`.  
Use `F.element_at(F.col("sorted_products"), 1)` to get the first product alphabetically.  
**Columns:** `franchiseID`, `name`, `sorted_products`, `first_alphabetical_product`


In [ ]:
result_6 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 6 ─────────────────
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'franchiseid' in cols, "Missing: franchiseID"
assert 'sorted_products' in cols, "Missing: sorted_products"
assert 'first_alphabetical_product' in cols, "Missing: first_alphabetical_product"
cnt = result_6.count()
assert cnt > 0, "Got 0 rows"
# first_alphabetical_product should equal element_at(sorted_products, 1)
check = result_6.filter(
    F.col("first_alphabetical_product") != F.element_at(F.col("sorted_products"), 1)
)
assert check.count() == 0, "first_alphabetical_product does not match first element of sorted array"
print(f"Problem 6 passed ✓  ({cnt} rows)")


## Problem 7 - Flatten Nested Arrays

On `wanderbricks.reviews`, collect review comments per `property_id` grouped by month.  
This gives an array of comment arrays (one per month). Use `F.flatten()` to combine all monthly arrays into one flat array per property.  
Add `total_comment_count` using `F.size()`.  
**Columns:** `property_id`, `monthly_comment_arrays`, `all_comments_flat`, `total_comment_count`


In [ ]:
result_7 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 7 ─────────────────
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'property_id' in cols, "Missing: property_id"
assert 'monthly_comment_arrays' in cols, "Missing: monthly_comment_arrays"
assert 'all_comments_flat' in cols, "Missing: all_comments_flat"
assert 'total_comment_count' in cols, "Missing: total_comment_count"
cnt = result_7.count()
assert cnt > 0, "Got 0 rows"
# total_comment_count should match size of flat array
mismatch = result_7.filter(F.col("total_comment_count") != F.size(F.col("all_comments_flat")))
assert mismatch.count() == 0, "total_comment_count does not match size of all_comments_flat"
print(f"Problem 7 passed ✓  ({cnt} rows)")
